In [1]:
import pandas as pd
import numpy as np
import os

os.makedirs("data/00-raw", exist_ok=True)
os.makedirs("data/01-interim", exist_ok=True)
os.makedirs("data/02-processed", exist_ok=True)

url = "https://raw.githubusercontent.com/the-pudding/data/refs/heads/master/birthday-effect/birthdays.csv"
local_path = "data/00-raw/birthdays.csv"

df = pd.read_csv(url)
df.to_csv(local_path, index=False)
df

,birth,death,age_floor,days_from_birthday,sex,marital,manner
0,1988-11-22,1990-09-26,1,-57,f,s,n
1,1988-02-09,1990-01-07,1,-33,f,s,n
2,1988-04-01,1990-02-13,1,-47,m,s,n
3,1988-04-11,1990-02-24,1,-46,f,s,n
4,1988-08-14,1990-02-27,1,-168,f,s,n
...,...,...,...,...,...,...,...
1955583,1903-01-15,2016-10-02,113,-105,f,w,n
1955584,1910-02-18,2023-09-19,113,-152,f,s,n
1955585,1911-02-28,2024-11-16,113,-104,f,w,n
1955586,1885-11-19,1999-12-02,114,13,NaN,w,n


### Tidying Dataset

Not too much to do because the dataset is already tidy; each row represents only one individual record, each column represents one single variable, and each cell contains one value. No further tidying necesary.

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1955588 entries, 0 to 1955587
Data columns (total 7 columns):
 #   Column              Dtype 
---  ------              ----- 
 0   birth               object
 1   death               object
 2   age_floor           int64 
 3   days_from_birthday  int64 
 4   sex                 object
 5   marital             object
 6   manner              object
dtypes: int64(2), object(5)
memory usage: 104.4+ MB


### Dataset Size

In [3]:
print("Shape:", df.shape)
print("Total rows:", df.shape[0])
print("Total columns:", df.shape[1])

Shape: (1955588, 7)
Total rows: 1955588
Total columns: 7


### Keep only important columns

We are only looking at age, marital status, sex, and manner of death in our study so we will drop extra columns to declutter

In [4]:
df = df[["age_floor", "sex", "marital", "manner"]]
df

,age_floor,sex,marital,manner
0,1,f,s,n
1,1,f,s,n
2,1,m,s,n
3,1,f,s,n
4,1,f,s,n
...,...,...,...,...
1955583,113,f,w,n
1955584,113,f,s,n
1955585,113,f,w,n
1955586,114,NaN,w,n


### Check for and clean Missing Data

    The dataset contains minimal missing data in most core variables (birth, death, age_floor). Marital status and manner contain <1% missing values. However, sex shows a 41% missing rate. 

In [5]:
missing_counts = df.isna().sum()
missing_percent = df.isna().mean() * 100

missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percent": missing_percent
}).sort_values("missing_percent", ascending=False)

missing_summary

,missing_count,missing_percent
sex,822646,42.066427
manner,10148,0.518923
marital,1627,0.083197
age_floor,0,0.000000


We don't want to drop 41% of our data, and it is likely that the missingness is not completely random (Women might choose to decline to state sex more often). Instead, we decided to fill missing sex values with d (declined to state), so we can see if declining to state sex has an effect. Other than that, missingness in the marital variable appears to be extremely low across all manner categories. The variation between groups is minimal, and several categories show no missing values at all. Because the missing rates are both very small and relatively similar across groups, there is no strong evidence that missingness is systematically associated with manner of death. This suggests that missing marital data is likely missing at random rather than structurally biased. Missing data from these categories will be dropped.

In [6]:
df_clean = df.fillna({"sex":"d"}, inplace = False)
df_clean = df_clean.dropna()

missing_counts = df_clean.isna().sum()
missing_percent = df_clean.isna().mean() * 100

missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percent": missing_percent
}).sort_values("missing_percent", ascending=False)

missing_summary


,missing_count,missing_percent
age_floor,0,0.0
sex,0,0.0
marital,0,0.0
manner,0,0.0


### Outliers and Suspicious Entries

In [7]:
flag_age = (df_clean["age_floor"] < 0) | (df_clean["age_floor"] > 90)
flag_sex = (((df_clean["sex"] == "m") | (df_clean["sex"] == "f")) | (df_clean["sex"] == "d")).apply(lambda x: not x)

marital = df_clean["marital"]
flag_marital = (((marital == "m") | (marital == "s")) | ((marital == "d") | (marital == "w"))).apply(lambda x: not x)

manner = df_clean["manner"]
def manner_checker(val):
    for letter in ["nashcpt"]:
        # n, a, s, h, c, p, t are the indicators for manner of death (n = natural, a = accident, etc)
        if val == letter:
            return False
flag_manner = manner.apply(manner_checker)

print(f"Weird Age values: {flag_age.sum()}")
print(f"Weird Sex values: {flag_sex.sum()}")
print(f"Weird Marital values: {flag_marital.sum()}")
print(f"Weird Manner values: {flag_manner.sum()}")

Weird Age values: 280207
Weird Sex values: 6
Weird Marital values: 4661
Weird Manner values: 0


Ages are all between 0 and 90, all manners of death are described, and there are only 4 rows that have a value of sex other than m, f, or d
However, marital has a little under 2000 entries that are something not described by our dataset

In [8]:
df_clean["marital"].value_counts()

marital
w    725798
m    705505
s    286984
d    220938
u      2343
a      2318
Name: count, dtype: int64

1778 values in the marital counts column have a value of "a" or "u", which were not described in the github we accessed the dataset from. However, 1778 is less than a percent of our data, so we will just drop the rows, alongside the 4 weird rows in the sex column

In [9]:
df_clean = df_clean[((df_clean["marital"] != "a") & (df_clean["marital"] != "u")) & (df_clean["sex"] != "u")]

Now that we've cleaned weird data, let's save our processed work

In [10]:
df_clean.to_csv("data/02-processed/birthdays_processed.csv", index=False)

print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)

Original shape: (1955588, 4)
Cleaned shape: (1939219, 4)


### Summary of Statistics

#### Age at Death

In [11]:
df_clean["age_floor"].describe()

count    1.939219e+06
mean     7.535524e+01
std      1.625929e+01
min      1.000000e+00
25%      6.700000e+01
50%      7.900000e+01
75%      8.700000e+01
max      1.140000e+02
Name: age_floor, dtype: float64

Age_floor refers to age at death, with the mean age at death being 75.1 and the median being 79. A higher median indicates a slight left skew, meaning there are younger deaths pulling the mean downward.

The interquartile range spans from 67 years (25th percentile) to 87 years (75th percentile), indicating that the middle 50% of individuals died between ages 67 and 87. This suggests the dataset is heavily concentrated among older adults, which aligns with expected mortality patterns.

The minimum recorded age is 1 year, and the maximum is 114 years, both of which fall within plausible biological limits. There are no obvious extreme outliers (e.g., negative ages or values above 120 after cleaning), suggesting the age variable appears valid and well-behaved.

Overall, the distribution of age at death appears realistic and consistent with known mortality patterns.

#### Manner of Death

In [12]:
df_clean["manner"].value_counts()

manner
n    1833568
a      77103
s      19097
h       6296
c       1576
p        973
t        606
Name: count, dtype: int64

Manner of death is largely made up of 'n' (natural causes). This means that any birthday effect will be primarily impacted by natural cause deaths.

#### Marital Status

In [13]:
df_clean["marital"].value_counts()

marital
w    725797
m    705502
s    286982
d    220938
Name: count, dtype: int64

Marital Status is still made up mostly of married people, with single being close behind, with a much smaller amount of divorcees and widows.

### Conclusions

The summary statistics indicate that:

- The dataset is large and robust (~1.96 million records).

- Age at death is concentrated among older adults and appears biologically plausible.

- The days_from_birthday variable is symmetrically distributed around zero, suggesting proper construction.

- Most deaths are due to natural causes, consistent with general mortality patterns.

- There are no obvious extreme outliers or structural anomalies in these core variables after cleaning.

The dataset appears suitable for further analysis of potential clustering of deaths around birthdays.